In [83]:
# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.competition_download('competitive-data-science-predict-future-sales')

print("Path to competition files:", path)

Path to competition files: C:\Users\Кирилл\.cache\kagglehub\competitions\competitive-data-science-predict-future-sales


In [84]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sales_train = pd.read_csv(os.path.join(path, 'sales_train.csv'))
items = pd.read_csv(os.path.join(path, 'items.csv'))
item_categories = pd.read_csv(os.path.join(path, 'item_categories.csv'))
sample_submission = pd.read_csv(os.path.join(path, 'sample_submission.csv'))
shops = pd.read_csv(os.path.join(path, 'shops.csv'))

sales_train.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,02.01.2013,0,59,22154,999.00,1.0
1,03.01.2013,0,25,2552,899.00,1.0
2,05.01.2013,0,25,2552,899.00,-1.0
3,06.01.2013,0,25,2554,1709.05,1.0
4,15.01.2013,0,25,2555,1099.00,1.0


In [85]:
sales_train = sales_train.merge(items, on='item_id', how='left')
sales_train = sales_train.drop('item_name', axis=1)

cols = ['date', 'date_block_num', 'shop_id', 'item_id', 'item_category_id', 'item_price', 'item_cnt_day']
sales_train = sales_train[cols]

sales_train.head()

,date,date_block_num,shop_id,item_id,item_category_id,item_price,item_cnt_day
0,02.01.2013,0,59,22154,37,999.00,1.0
1,03.01.2013,0,25,2552,58,899.00,1.0
2,05.01.2013,0,25,2552,58,899.00,-1.0
3,06.01.2013,0,25,2554,58,1709.05,1.0
4,15.01.2013,0,25,2555,56,1099.00,1.0


In [86]:
grouped_sales = sales_train.groupby(['date_block_num', 'shop_id', 'item_id']).agg({'item_cnt_day': ['sum']}).reset_index()
grouped_sales = grouped_sales.rename(columns={'item_cnt_day': 'item_cnt_month'})
grouped_sales['item_cnt_month'] = grouped_sales['item_cnt_month'].clip(0, 20)
grouped_sales.head()

,date_block_num,shop_id,item_id,item_cnt_month
,,,,sum
0,0,0,32,6.0
1,0,0,33,3.0
2,0,0,35,1.0
3,0,0,43,1.0
4,0,0,51,2.0


In [87]:
# кол-во продаж за предыдущий месяц, 6 месяцев и 12 месяцев
grouped_sales['item_cnt_lag_1'] = grouped_sales.groupby(['shop_id', 'item_id'])['item_cnt_month'].shift(1)
grouped_sales['item_cnt_lag_6'] = grouped_sales.groupby(['shop_id', 'item_id'])['item_cnt_month'].shift(6)
grouped_sales['item_cnt_lag_12'] = grouped_sales.groupby(['shop_id', 'item_id'])['item_cnt_month'].shift(12)

# там где данных нет 0
grouped_sales['item_cnt_lag_1'] = grouped_sales['item_cnt_lag_1'].fillna(0)
grouped_sales['item_cnt_lag_6'] = grouped_sales['item_cnt_lag_6'].fillna(0)
grouped_sales['item_cnt_lag_12'] = grouped_sales['item_cnt_lag_12'].fillna(0)

# удаляю первый год, так как там нет данных за предыдущий год
grouped_sales = grouped_sales[grouped_sales['date_block_num'] >= 12]

In [88]:
grouped_sales

,date_block_num,shop_id,item_id,item_cnt_month,item_cnt_lag_1,item_cnt_lag_6,item_cnt_lag_12
,,,,sum,,,
687724,12,2,32,1.0,0.0,0.0,0.0
687725,12,2,33,1.0,1.0,0.0,0.0
687726,12,2,99,1.0,0.0,0.0,0.0
687727,12,2,482,2.0,1.0,2.0,0.0
687728,12,2,485,1.0,1.0,2.0,0.0
...,...,...,...,...,...,...,...
1609119,33,59,22087,6.0,3.0,2.0,6.0
1609120,33,59,22088,2.0,1.0,3.0,7.0
1609121,33,59,22091,1.0,3.0,1.0,1.0
